# Entity Resolution — Gradient Boosting Pipeline

End-to-end notebook that calls modular functions from `src/`.

**Sections:**
1. Configuration
2. Load database
3. Inspect data
4. Candidate-generation statistics
5. Generate training pairs
6. Feature engineering
7. Train LightGBM
8. Evaluate LightGBM
9. Train XGBoost
10. Evaluate XGBoost
11. Threshold tuning
12. Error analysis
13. Save models
14. Generate test predictions

## 1. Configuration

In [1]:
import sys, os

# Ensure project root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import (
    DB_PATH, RANDOM_STATE, NEGATIVE_TO_POSITIVE_RATIO,
    LGBM_PARAMS, XGB_PARAMS, VAL_FRACTION, MODEL_DIR, OUTPUT_DIR,
)

print(f"DB Path:          {DB_PATH}")
print(f"Random State:     {RANDOM_STATE}")
print(f"Neg:Pos Ratio:    {NEGATIVE_TO_POSITIVE_RATIO}")
print(f"Val Fraction:     {VAL_FRACTION}")
print(f"Model Dir:        {MODEL_DIR}")
print(f"Output Dir:       {OUTPUT_DIR}")

DB Path:          C:\Users\shara\Desktop\Amazon_ml\database\amazon_ml.db
Random State:     42
Neg:Pos Ratio:    2
Val Fraction:     0.2
Model Dir:        C:\Users\shara\Desktop\Amazon_ml\models
Output Dir:       C:\Users\shara\Desktop\Amazon_ml\output


## 2. Load Database

In [2]:
from src.database import get_connection, setup_database, count_rows

conn = get_connection()

# This is idempotent — skips columns/indexes that already exist
setup_database(conn, progress=True)

Setting up database...

--- source1 ---
  source1: all normalized columns already exist, skipping.
  Creating index idx_source1_name_translit...
  Creating index idx_source1_name_no_legal...
  Creating index idx_source1_name_compact...
  Creating index idx_source1_first_last...
  Creating index idx_source1_country_addr...

--- source2 ---
  source2: populating 6 columns (name_norm, name_translit, name_no_legal, name_compact, addr_norm, name_first_last)...
  source2: done — 5,034,616 rows updated.
  Creating index idx_source2_name_translit...
  Creating index idx_source2_name_no_legal...
  Creating index idx_source2_name_compact...
  Creating index idx_source2_first_last...
  Creating index idx_source2_country_addr...

--- source3 ---
  source3: populating 6 columns (name_norm, name_translit, name_no_legal, name_compact, addr_norm, name_first_last)...
  source3: done — 5,285,603 rows updated.
  Creating index idx_source3_name_translit...
  Creating index idx_source3_name_no_legal...
  C

## 3. Inspect Data

In [3]:
for table in ["source1", "source2", "source3", "ground_truth", "ground_truth_pairs"]:
    try:
        n = count_rows(conn, table)
        print(f"  {table}: {n:,} rows")
    except Exception as e:
        print(f"  {table}: {e}")

# Sample S1 entity
print("\nSample Source-1 entity:")
row = conn.execute("SELECT * FROM source1 LIMIT 1").fetchone()
cols = [d[1] for d in conn.execute("PRAGMA table_info(source1)").fetchall()]
for c, v in zip(cols, row):
    print(f"  {c}: {v}")

  source1: 2,206,821 rows
  source2: 5,034,616 rows
  source3: 5,285,603 rows
  ground_truth: 2,206,821 rows
  ground_truth_pairs: 7,638,365 rows

Sample Source-1 entity:
  entity_id: S1-925783039
  business_name: Orelee's Barbershop
  business_address: 1795 Westchester Drive, High Point, NC
  country: US
  country_norm: us
  name_prefix: orel
  address_prefix: 1795
  name_norm: orelees barbershop
  name_translit: orelees barbershop
  name_no_legal: orelees barbershop
  name_compact: oreleesbarbershop
  addr_norm: 1795 westchester drive high point nc
  name_first_last: orelees|barbershop


## 4. Candidate-Generation Statistics

In [ ]:
from src.candidate_generation import (
    fetch_s1_blocking_data, generate_candidates_batch, measure_candidate_recall,
)

# Evaluate candidate recall on a sample of S1 entities
SAMPLE_SIZE = 500  # Use a small sample for quick inspection; increase later

sample_s1 = fetch_s1_blocking_data(conn, limit=SAMPLE_SIZE)
print(f"Fetched {len(sample_s1)} S1 entities for candidate generation sample.")

candidates_dict = generate_candidates_batch(conn, sample_s1, progress=False)

n_cands = [len(v) for v in candidates_dict.values()]
print(f"Candidates per S1 — min: {min(n_cands)}, max: {max(n_cands)}, "
      f"mean: {sum(n_cands)/len(n_cands):.1f}, total: {sum(n_cands):,}")

recall_info = measure_candidate_recall(conn, candidates_dict, progress=True)

Fetched 500 S1 entities for candidate generation sample.


## 5. Generate Training Pairs

Split S1 entities into train/val **at the entity level** to prevent leakage.

In [ ]:
from src.training_data import (
    get_s1_train_val_split, generate_training_data, validate_training_data,
)

train_s1_ids, val_s1_ids = get_s1_train_val_split(conn)
print(f"Train S1 entities: {len(train_s1_ids):,}")
print(f"Val   S1 entities: {len(val_s1_ids):,}")
print(f"Overlap:           {len(set(train_s1_ids) & set(val_s1_ids))}")

## 6. Feature Engineering

Generate feature matrices for train and validation sets.

In [ ]:
# Set max_s1=None for full run; use a small number for smoke testing
MAX_S1 = None  # e.g. 200 for a quick test

print("Generating TRAINING data...")
X_train, y_train, train_pair_ids = generate_training_data(
    conn, train_s1_ids, progress=True, max_s1=MAX_S1,
)

print("\nGenerating VALIDATION data...")
X_val, y_val, val_pair_ids = generate_training_data(
    conn, val_s1_ids, progress=True, max_s1=MAX_S1,
)

In [ ]:
from src.features import FEATURE_NAMES

validate_training_data(X_train, y_train, train_pair_ids, train_s1_ids, val_s1_ids)
validate_training_data(X_val, y_val, val_pair_ids, val_s1_ids, train_s1_ids)

## 7. Train LightGBM

In [ ]:
from src.train_lgbm import train_lightgbm, save_lightgbm

lgbm_model = train_lightgbm(X_train, y_train, X_val, y_val, progress=True)

## 8. Evaluate LightGBM

In [ ]:
from src.evaluate import evaluate_model

lgbm_results = evaluate_model(
    lgbm_model, X_val, y_val, val_pair_ids,
    model_name="LightGBM", progress=True,
)

lgbm_threshold = lgbm_results["best_threshold"]
print(f"\nSelected LightGBM threshold: {lgbm_threshold:.3f}")

## 9. Train XGBoost

In [ ]:
from src.train_xgb import train_xgboost, save_xgboost

xgb_model = train_xgboost(X_train, y_train, X_val, y_val, progress=True)

## 10. Evaluate XGBoost

In [ ]:
xgb_results = evaluate_model(
    xgb_model, X_val, y_val, val_pair_ids,
    model_name="XGBoost", progress=True,
)

xgb_threshold = xgb_results["best_threshold"]
print(f"\nSelected XGBoost threshold: {xgb_threshold:.3f}")

## 11. Threshold Tuning — Model Comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, results, name in [
    (axes[0], lgbm_results, "LightGBM"),
    (axes[1], xgb_results, "XGBoost"),
]:
    all_r = results["threshold_results"]["all_results"]
    ts = [r["threshold"] for r in all_r]
    f05s = [r["f05"] for r in all_r]
    precs = [r["precision"] for r in all_r]
    recs = [r["recall"] for r in all_r]

    ax.plot(ts, f05s, label="F0.5", linewidth=2)
    ax.plot(ts, precs, label="Precision", linestyle="--")
    ax.plot(ts, recs, label="Recall", linestyle="--")
    best_t = results["best_threshold"]
    ax.axvline(best_t, color="red", linestyle=":", label=f"Best t={best_t:.3f}")
    ax.set_title(f"{name} — Threshold vs Metrics")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Score")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Comparison table
print(f"\n{'Model':<12} {'Threshold':>10} {'Pair F0.5':>10} {'Entity F0.5':>12} {'Precision':>10} {'Recall':>8}")
print("-" * 66)
for res, name in [(lgbm_results, "LightGBM"), (xgb_results, "XGBoost")]:
    b = res["threshold_results"]["best"]
    e = res["entity_results"]
    print(f"{name:<12} {b['threshold']:>10.3f} {b['f05']:>10.4f} {e['macro_f05']:>12.4f} {b['precision']:>10.4f} {b['recall']:>8.4f}")

## 12. Error Analysis

In [ ]:
from src.evaluate import error_analysis

# Use the better-performing model for error analysis
best_model_name = "LightGBM" if lgbm_results["entity_results"]["macro_f05"] >= xgb_results["entity_results"]["macro_f05"] else "XGBoost"
best_model = lgbm_model if best_model_name == "LightGBM" else xgb_model
best_threshold = lgbm_threshold if best_model_name == "LightGBM" else xgb_threshold

print(f"Error analysis using {best_model_name} (threshold={best_threshold:.3f})\n")

errors = error_analysis(
    best_model, X_val, y_val, val_pair_ids, conn,
    threshold=best_threshold, max_errors=30, progress=True,
)

In [ ]:
# Show top false positives
print("=" * 80)
print("TOP FALSE POSITIVES (predicted match, but ground truth says no)")
print("=" * 80)
for i, fp in enumerate(errors["false_positives"][:10]):
    print(f"\n--- FP #{i+1} (prob={fp['proba']:.4f}) ---")
    print(f"  S1:   {fp.get('s1_name', '?')}  |  {fp.get('s1_address', '?')}  |  {fp.get('s1_country', '?')}")
    print(f"  Cand: {fp.get('cand_name', '?')}  |  {fp.get('cand_address', '?')}  |  {fp.get('cand_country', '?')}")
    top_feats = sorted(fp["features"].items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"  Top features: {', '.join(f'{k}={v:.3f}' for k, v in top_feats)}")

In [ ]:
# Show top false negatives
print("=" * 80)
print("TOP FALSE NEGATIVES (missed true match)")
print("=" * 80)
for i, fn in enumerate(errors["false_negatives"][:10]):
    print(f"\n--- FN #{i+1} (prob={fn['proba']:.4f}) ---")
    print(f"  S1:   {fn.get('s1_name', '?')}  |  {fn.get('s1_address', '?')}  |  {fn.get('s1_country', '?')}")
    print(f"  Cand: {fn.get('cand_name', '?')}  |  {fn.get('cand_address', '?')}  |  {fn.get('cand_country', '?')}")
    top_feats = sorted(fn["features"].items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"  Top features: {', '.join(f'{k}={v:.3f}' for k, v in top_feats)}")

## 13. Save Models

In [ ]:
import json

print("Saving LightGBM...")
save_lightgbm(lgbm_model)

print("\nSaving XGBoost...")
save_xgboost(xgb_model)

# Save the selected threshold
MODEL_DIR.mkdir(parents=True, exist_ok=True)
threshold_info = {
    "best_model": best_model_name,
    "lgbm_threshold": lgbm_threshold,
    "xgb_threshold": xgb_threshold,
    "lgbm_pair_f05": lgbm_results["threshold_results"]["best"]["f05"],
    "xgb_pair_f05": xgb_results["threshold_results"]["best"]["f05"],
    "lgbm_entity_f05": lgbm_results["entity_results"]["macro_f05"],
    "xgb_entity_f05": xgb_results["entity_results"]["macro_f05"],
}
with open(MODEL_DIR / "threshold_info.json", "w") as f:
    json.dump(threshold_info, f, indent=2)
print(f"\nSaved threshold info to {MODEL_DIR / 'threshold_info.json'}")

## 14. Generate Test Predictions

Use the best model and its optimal threshold to produce a submission file.

In [ ]:
from src.inference import run_inference

output_path = run_inference(
    conn,
    model=best_model,
    threshold=best_threshold,
    progress=True,
)

print(f"\nSubmission file written to: {output_path}")

In [ ]:
# Quick sanity check on the submission
import pandas as pd

sub = pd.read_csv(output_path)
print(f"Submission shape: {sub.shape}")
print(f"\nSample rows:")
print(sub.head(10).to_string())

non_empty = sub["matched_entity_ids"].fillna("").str.strip().ne("").sum()
print(f"\nEntities with >= 1 match: {non_empty:,} / {len(sub):,}")

In [ ]:
conn.close()
print("Done.")